# One 4-bit datapath — faithful harness test (Colab)

Drives the **real** `FP_Quantization_Experiments` package from your `Thesis_Compression` repo — the actual
`quantize_model_fp` (Hadamard rotation + **Hessian weight reconstruction** → NVFP4/E2M1 weights) and the
real GF4 activation quantizers. No reimplementation of the quant path.

**Thesis under test — no second datapath.** The MLP-output layers (`fc2`/`down_proj`) are normally kept in
**FP16** because their per-token activation outliers collapse single-pass 4-bit — that FP16 path *is* a
second datapath. We test whether running those layers **entirely in 4-bit** with extra **residual GF4
passes** matches FP16 retention, so the whole model is one 4-bit datapath.

Per model we build two faithful quantizations and compare WikiText-2 PPL:

| profile | build | bulk acts | outlier acts | datapath |
|---|---|---|---|---|
| **retain** (baseline) | outliers skipped→FP16 | 2-pass GF4 | **FP16** | **two** |
| A16-all | all quantized | FP16 | FP16 | (acts 16-bit) |
| 1dp out-1p | all quantized | 2-pass | 1-pass GF4 | one |
| 1dp out-2p | all quantized | 2-pass | 2-pass GF4 | one |
| 1dp out-4p | all quantized | 2-pass | 4-pass GF4 | one |
| 1dp out-8p | all quantized | 2-pass | 8-pass GF4 | one |

**Claim holds** if a *1dp* row ≈ *retain*. Find the smallest outlier pass-count that gets there, and how it
scales. `lm_head` stays FP16 (default head handling). Local sanity (self-contained proxy) already showed
2-pass outliers ≈ FP16 retain on OPT-125m; this notebook confirms it with the real harness at scale.

## 1. Clone the real package + install deps  (Runtime → GPU)

In [ ]:
import os
# Must precede any CUDA-extension JIT (matches the harness header). Covers T4/L4/A100/4090.
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "7.5;8.0;8.6;8.9")

REPO = "https://github.com/jvap2/Thesis_Compression.git"
PKG_SUBDIR = "Python_Jenks_Test/Jenks_Tests"        # holds FP_Quantization_Experiments/

# Sparse-checkout only the package dir (repo is large). CV extras are import-guarded.
!rm -rf Thesis_Compression
!git clone --depth 1 --filter=blob:none --sparse {REPO}
%cd Thesis_Compression
!git sparse-checkout set {PKG_SUBDIR}/FP_Quantization_Experiments
%cd ..
!pip -q install "transformers>=4.44" "datasets>=2.20" accelerate scipy
print("cloned + deps ready")

## 2. Ensure the N-pass activation mode exists, then import

The harness ships `gf4_residual` (2-pass). This idempotent patch adds a general **`gf4_residualN`** mode
(reads `module.gf4_npass`) so we can run the outlier layers at 4 or 8 passes. It's a no-op once your repo
has it (e.g. after you merge the change).

In [ ]:
import sys, pathlib
PKG_ROOT = str(pathlib.Path("Thesis_Compression") / "Python_Jenks_Test" / "Jenks_Tests")
sys.path.insert(0, PKG_ROOT)
BSP = pathlib.Path(PKG_ROOT) / "FP_Quantization_Experiments" / "bit_split.py"

src = BSP.read_text(); changed = False
if "def quantize_activations_gf4_npass" not in src:
    anchor = ('    x_q2 = quantize_activations_gf4(residual, block_size, clip_ratio=clip_ratio2, levels=levels)\n'
              '    return (x_q1 + x_q2).to(x.dtype)')
    assert anchor in src, "npass anchor not found — repo layout changed"
    src = src.replace(anchor, anchor +
        '\n\n\ndef quantize_activations_gf4_npass(x, block_size, n_pass=4, clip_ratio=2.5, levels=None):\n'
        '    # N-stage residual GF4 (generalizes the 2-stage residual)\n'
        '    xf = x.float(); q = torch.zeros_like(xf)\n'
        '    for _ in range(int(n_pass)):\n'
        '        q = q + quantize_activations_gf4(xf - q, block_size, clip_ratio=clip_ratio, levels=levels)\n'
        '    return q.to(x.dtype)', 1); changed = True
if "gf4_residualN" not in src:
    fbr = ('                elif self._act_quant_mode == "gf4_residual":\n'
           '                    clip = self.act_clip_ratio if self.act_clip_ratio is not None else 2.5\n'
           '                    x_2d = quantize_activations_gf4_residual(\n'
           '                        x_2d, bs, clip_ratio1=clip, clip_ratio2=clip, levels=lvl\n'
           '                    ).to(orig_dtype)')
    assert fbr in src, "forward-branch anchor not found — repo layout changed"
    src = src.replace(fbr, fbr +
        '\n\n                elif self._act_quant_mode == "gf4_residualN":\n'
        '                    clip = self.act_clip_ratio if self.act_clip_ratio is not None else 2.5\n'
        '                    x_2d = quantize_activations_gf4_npass(\n'
        '                        x_2d, bs, n_pass=getattr(self, "gf4_npass", 4),\n'
        '                        clip_ratio=clip, levels=lvl\n'
        '                    ).to(orig_dtype)', 1); changed = True
if changed: BSP.write_text(src); print("patched gf4_residualN into bit_split.py")
else: print("gf4_residualN already present")

import torch, torch.nn as nn, math, gc, time
from FP_Quantization_Experiments import (
    quantize_model_fp, act_quant_mode, QuantLinearFP, HadamardQuantLinearFP,
    LLAMA_SKIP_PATTERNS,
)
print("imported real package OK")

In [ ]:
# Gated models only (Llama). Nothing saved to disk.
try:
    from huggingface_hub import notebook_login
    # notebook_login()
except Exception as e:
    print(e)

## 3. Config

In [ ]:
CONFIG = dict(
    BLOCK_SIZE = 16, E_BITS = 2, M_BITS = 1, E_SCALE = 4, M_SCALE = 3,   # NVFP4 / E2M1 (harness line 745)
    HAD_BLOCK = "auto",          # full-row Hadamard, as the harness
    SEQLEN = 2048, N_SAMPLES = 40, CALIB_SEQLEN = 512, N_CALIB_BATCHES = 4,
    LEAN = True,                 # free per-layer originals (fits 6.7B/7B on 24GB)
    OFFLOAD = False,             # block-sequential CPU-offload calib (set True for 13B+/24GB)
    BULK_MODE = "gf4_residual",  # bulk activation treatment (2-pass) — held fixed
    OUTLIER_SWEEP = [("1dp out-1p", "gf4", None),
                     ("1dp out-2p", "gf4_residual", None),
                     ("1dp out-4p", "gf4_residualN", 4),
                     ("1dp out-8p", "gf4_residualN", 8)],
    DO_RETAIN = True,            # also build the FP16-retain baseline (2x quantization cost)
)
MODELS = [
    "facebook/opt-1.3b",
    "facebook/opt-2.7b",
    # "facebook/opt-6.7b",           # LEAN=True fits ~24GB
    # "meta-llama/Llama-2-7b-hf",    # login above; down_proj is the outlier layer
    # "facebook/opt-13b",            # set OFFLOAD=True
]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def outlier_patterns(model_name):
    n = model_name.lower()
    if "llama" in n or "mistral" in n: return tuple(LLAMA_SKIP_PATTERNS)   # ("down_proj",)
    return ("fc2",)                                                        # OPT / GPT-2

## 4. Data + perplexity (non-overlapping sliding window = harness `compute_standard_ppl`)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

def load_wikitext(split):
    for repo in ("Salesforce/wikitext", "wikitext"):
        try: return load_dataset(repo, "wikitext-2-raw-v1", split=split)
        except Exception: pass
    raise RuntimeError("could not load wikitext-2")

def make_calib_loader(tokenizer, cfg):
    ds = load_wikitext("train")
    def collate(batch):
        texts = [x["text"] for x in batch if len(x["text"].strip()) > 0]
        if not texts: return None, None
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
        ids = tokenizer(texts, return_tensors="pt", padding=True,
                        truncation=True, max_length=cfg["CALIB_SEQLEN"]).input_ids
        return ids, ids
    return DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collate)

@torch.no_grad()
def eval_ppl(model, enc, seqlen, n_samples, device):
    model.eval()
    n = min(enc.shape[1] // seqlen, n_samples); nll = cnt = 0
    for i in range(n):
        ids = enc[:, i*seqlen:(i+1)*seqlen].to(device)
        out = model(ids, labels=ids)
        nll += out.loss.float().item() * (seqlen - 1); cnt += seqlen - 1
    return math.exp(nll / cnt)

## 5. Faithful build + per-layer activation modes

In [ ]:
def load_fp16(name, device):
    tok = AutoTokenizer.from_pretrained(name, use_fast=False)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to(device).eval()
    m.config.use_cache = False
    return m, tok

def build_quant(model, calib_loader, cfg, device, skip_patterns):
    # Faithful to FP_QuantNetworkTest_LLM.py line ~745: Hadamard + Hessian-reconstructed NVFP4 weights,
    # GF4 activations. skip_patterns=() -> quantize EVERYTHING (single datapath); =('fc2',) -> FP16 retain.
    return quantize_model_fp(
        model, calib_loader, block_size=cfg["BLOCK_SIZE"],
        e_bits=cfg["E_BITS"], m_bits=cfg["M_BITS"],
        e_bits_scale=cfg["E_SCALE"], m_bits_scale=cfg["M_SCALE"],
        device=device, use_HG=False, use_Hessian=False, use_adap=False, use_forward=False,
        Hadamard=True, joint=False, preshift=False, decompose=False,
        had_block_size=cfg["HAD_BLOCK"], use_gf4=True,
        num_calib_batches=cfg["N_CALIB_BATCHES"],
        extra_skip_patterns=skip_patterns, lean=cfg["LEAN"], offload=cfg["OFFLOAD"],
    )

def set_modes(model, bulk_mode, patterns, outlier_mode=None, outlier_npass=None):
    # bulk_mode / outlier_mode: None = A16 (FP16 acts) | 'gf4' | 'gf4_residual' | 'gf4_residualN'.
    # Outlier layers skipped to FP16 are not Quant modules, so they stay FP16 regardless.
    for name, mod in model.named_modules():
        if isinstance(mod, (QuantLinearFP, HadamardQuantLinearFP)):
            if outlier_mode is not None and any(p in name for p in patterns):
                if outlier_npass is not None: mod.gf4_npass = outlier_npass
                mod.act_quant_mode = outlier_mode
            else:
                mod.act_quant_mode = bulk_mode

## 6. Run the sweep

In [ ]:
import pandas as pd
rows = []
for name in MODELS:
    print(f"\n=== {name} ===")
    t0 = time.time(); pats = outlier_patterns(name)
    res = {"model": name}

    # reference FP16 PPL (unquantized)
    m, tok = load_fp16(name, DEVICE)
    enc = tok("\n\n".join(load_wikitext("test")["text"]), return_tensors="pt").input_ids
    res["fp16"] = eval_ppl(m, enc, CONFIG["SEQLEN"], CONFIG["N_SAMPLES"], DEVICE)
    print(f"  fp16 {res['fp16']:.3f}")
    calib = make_calib_loader(tok, CONFIG)

    # ---- Build A: FP16 outlier retention (two datapaths) ----
    if CONFIG["DO_RETAIN"]:
        qA = build_quant(m, calib, CONFIG, DEVICE, skip_patterns=pats)     # outliers -> FP16
        set_modes(qA, CONFIG["BULK_MODE"], pats, outlier_mode=None)        # bulk 2-pass; outliers FP16
        res["retain (outFP16)"] = eval_ppl(qA, enc, CONFIG["SEQLEN"], CONFIG["N_SAMPLES"], DEVICE)
        print(f"  retain (2 datapaths) {res['retain (outFP16)']:.3f}")
        del qA; gc.collect(); torch.cuda.empty_cache()
        m, tok = load_fp16(name, DEVICE)                                   # fresh model for build B
        calib = make_calib_loader(tok, CONFIG)

    # ---- Build B: single 4-bit datapath (quantize EVERYTHING) ----
    qB = build_quant(m, calib, CONFIG, DEVICE, skip_patterns=())
    set_modes(qB, None, pats, outlier_mode=None)                           # A16-all reference
    res["A16 (acts FP16)"] = eval_ppl(qB, enc, CONFIG["SEQLEN"], CONFIG["N_SAMPLES"], DEVICE)
    print(f"  A16 (all acts FP16) {res['A16 (acts FP16)']:.3f}")
    for label, omode, onp in CONFIG["OUTLIER_SWEEP"]:
        set_modes(qB, CONFIG["BULK_MODE"], pats, outlier_mode=omode, outlier_npass=onp)
        res[label] = eval_ppl(qB, enc, CONFIG["SEQLEN"], CONFIG["N_SAMPLES"], DEVICE)
        base = res.get("retain (outFP16)")
        d = f"  (dP vs retain {res[label]-base:+.3f})" if base else ""
        print(f"  {label}: {res[label]:.3f}{d}")
    del qB, m; gc.collect(); torch.cuda.empty_cache()

    res["minutes"] = round((time.time()-t0)/60, 1)
    rows.append(res); pd.DataFrame(rows).to_csv("gf4_datapath_results.csv", index=False)

df = pd.DataFrame(rows); df

## 7. Plot + summary

In [ ]:
import matplotlib.pyplot as plt
if len(rows):
    cols = [c for c in df.columns if c not in ("model", "fp16", "minutes")]
    fig, ax = plt.subplots(figsize=(2.0*len(rows)+4, 4.4))
    x = range(len(df)); w = 0.8/len(cols)
    for j, c in enumerate(cols):
        ax.bar([i + (j-len(cols)/2)*w + w/2 for i in x], df[c], w, label=c)
    ax.set_xticks(list(x)); ax.set_xticklabels([m.split('/')[-1] for m in df["model"]], rotation=15, ha="right")
    ax.set_ylabel("WikiText-2 PPL"); ax.set_title("Single 4-bit datapath vs FP16 outlier retention (real harness)")
    ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("gf4_datapath.png", dpi=140); plt.show()
    if "retain (outFP16)" in df:
        print("\nΔ vs retain (near-zero = single 4-bit datapath matches the FP16 unit):")
        for _, r in df.iterrows():
            s = "  ".join(f"{c.split()[-1]}:{r[c]-r['retain (outFP16)']:+.2f}" for c in cols if c.startswith("1dp"))
            print(f"  {r['model'].split('/')[-1]:<18} retain={r['retain (outFP16)']:.3f} | {s}")

## Notes

- **Faithful:** weights are your real Hadamard + **Hessian-reconstructed** NVFP4 (via `quantize_model_fp`,
  same args as `FP_QuantNetworkTest_LLM.py`); activations use the real `gf4` / `gf4_residual` /
  `gf4_residualN` quantizers. Only the calib loader + PPL loop are standard (they match the harness method).
- **Headline number:** the smallest outlier pass-count whose `1dp` row ≈ `retain` — that many GF4 passes on
  `fc2`/`down_proj` replaces their FP16 unit, giving a single 4-bit datapath. Watch it grow with scale.
- **Memory:** `LEAN=True` fits 6.7B/7B on 24 GB; set `OFFLOAD=True` for 13B+. `DO_RETAIN=False` skips the
  second (retain) build if you only need the single-datapath rows.
- **Merge upstream:** once you add `gf4_residualN` to `bit_split.py` in the repo, the patch cell is a no-op.
- Set `MODEL`-specific outlier via `outlier_patterns`; add `lm_head` there to fold the head in too.
